# Pseudonymizer - fine-tune on Colab

Before running: **Runtime -> Change runtime type -> A100 GPU** (Colab Pro). Enable *Background execution* for long runs.

Data (`train.parquet`, `dev.parquet`, `dev.jsonl`, optionally `dev_judges.*`) must be in Google Drive at `MyDrive/ai-camp/` (or `MyDrive/ai-camp/silver/`). Code is cloned from GitHub; every run is copied back to `MyDrive/ai-camp/runs/<name>/`.

Timing on A100, 24k docs x 3 epochs: base models ~25 min, large ~1-1.5 h.

In [ ]:
#@title 1. Code
REPO = "EnderLancer/anonimimizer"  #@param {type:"string"}
BRANCH = "dmelashych/datagen"  #@param {type:"string"}
import os
# private repo: add a GitHub token as Colab secret GH_TOKEN (left sidebar -> key icon -> enable notebook access)
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists('/content/ai-camp/train'):
    !rm -rf /content/ai-camp && git clone -q --branch "$BRANCH" "$url" /content/ai-camp
else:
    !cd /content/ai-camp && git pull -q
assert os.path.exists('/content/ai-camp/train'), 'clone failed: check REPO/BRANCH and the GH_TOKEN secret'
%cd /content/ai-camp/train
!pip install -q -e . 2>&1 | tail -1
import os; os.environ.update(TRANSFORMERS_VERBOSITY='error', PYTHONWARNINGS='ignore', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', gpu)
if 'A100' not in gpu and 'L4' not in gpu:
    print('WARNING: not an A100 - Runtime -> Change runtime type -> A100 GPU (Colab Pro); on T4 halve every batch size')

In [ ]:
#@title 2. Data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
from tqdm.auto import tqdm
DRIVE = '/content/drive/MyDrive/ai-camp'
os.makedirs('/content/ai-camp/data/silver', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)
src = f'{DRIVE}/silver' if os.path.exists(f'{DRIVE}/silver/train.parquet') else DRIVE
files = glob.glob(f'{src}/*.parquet') + glob.glob(f'{src}/*.jsonl')
assert files, f'no parquet/jsonl files found in {DRIVE} or {DRIVE}/silver'
for f in tqdm(files, desc='copy from Drive'):
    shutil.copy(f, '/content/ai-camp/data/silver/')
!ls -la /content/ai-camp/data/silver/
assert os.path.exists('/content/ai-camp/data/evaluate.py'), 'data/evaluate.py missing in the repo'
print('OK, data from', src)

In [ ]:
#@title 3. (optional) Hugging Face login - needed only for --push-to
PUSH_TO = ""  #@param {type:"string"}
if PUSH_TO:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get('HF_TOKEN'))  # add HF_TOKEN in Colab: left sidebar -> Secrets

In [ ]:
#@title 4. Train one model
MODEL = "benjamin/roberta-base-wechsel-ukrainian"  #@param ["benjamin/roberta-base-wechsel-ukrainian", "FacebookAI/xlm-roberta-base", "microsoft/mdeberta-v3-base", "benjamin/roberta-large-wechsel-ukrainian", "FacebookAI/xlm-roberta-large", "Goader/liberta-large-v2"] {allow-input: true}
RUN = "wechsel_base_full"  #@param {type:"string"}
EPOCHS = 3  #@param {type:"number"}
LR = 4e-5  #@param {type:"number"}  # base: 4e-5 at batch 64; large: 2e-5
BATCH = 64  #@param {type:"integer"}  # A100 40GB: base 64, large 32; T4 16GB: base 16, large 8 + GRAD_ACCUM 2
GRAD_ACCUM = 1  #@param {type:"integer"}
N_DOCS = 0  #@param {type:"integer"}
DEV_DOCS = 500  #@param {type:"integer"}
extra = (f' --n-docs {N_DOCS}' if N_DOCS else '') + (f' --push-to {PUSH_TO}' if PUSH_TO else '')
# --skip-if-done: if the same model + hyper-parameters + data version is already on Drive, nothing is retrained
!python scripts/train.py --model $MODEL --out runs/$RUN --epochs $EPOCHS --lr $LR --batch $BATCH --grad-accum $GRAD_ACCUM --dev-docs $DEV_DOCS --skip-if-done $DRIVE/runs/$RUN $extra
!rm -rf runs/$RUN/checkpoints && mkdir -p $DRIVE/runs/$RUN && cp -r runs/$RUN/. $DRIVE/runs/$RUN/ && echo saved to $DRIVE/runs/$RUN - model, config.json with data version and fingerprint, train.log

In [ ]:
#@title 5. Score on full dev with the official script + copy the run to Drive
!python scripts/predict.py --model runs/$RUN/model --input ../data/silver/dev.jsonl --output runs/$RUN/dev_sub.jsonl --batch 32
!python ../data/evaluate.py ../data/silver/dev.jsonl runs/$RUN/dev_sub.jsonl
import os
if os.path.exists('../data/silver/dev_judges.jsonl'):
    !python scripts/predict.py --model runs/$RUN/model --input ../data/silver/dev_judges.jsonl --output runs/$RUN/dev_judges_sub.jsonl --batch 32
    !python ../data/evaluate.py ../data/silver/dev_judges.jsonl runs/$RUN/dev_judges_sub.jsonl | head -5
!rm -rf runs/$RUN/checkpoints && cp -r runs/$RUN $DRIVE/runs/ && echo copied to $DRIVE/runs/$RUN

In [ ]:
#@title 6. (optional) CPU timing estimate - the organizers' limit is 2 h for 240 docs on CPU
!head -n 30 ../data/silver/dev.jsonl > /tmp/dev30.jsonl
import time; t0 = time.time()
!python scripts/predict.py --model runs/$RUN/model --input /tmp/dev30.jsonl --output /tmp/cpu_sub.jsonl --device cpu --threads 4 --batch 8
dt = (time.time() - t0) / 30
print(f'{dt:.2f} s/doc on Colab CPU -> test (240 docs) ~ {dt*240/60:.1f} min  (a laptop CPU is usually 1.5-2x faster)')

In [ ]:
#@title Report (Markdown + HTML) -> Drive, shown inline
import time, shutil
from IPython.display import HTML, display
stamp = time.strftime('%Y%m%d_%H%M')
ERRORS_RUN = ""  #@param {type:"string"}   # run to dump dev errors for; empty = best run
err = f' --errors-run {ERRORS_RUN}' if ERRORS_RUN else ''
!python scripts/report.py --runs runs --out runs/report.md $err > /dev/null
for ext in ('md', 'html'):
    shutil.copy(f'runs/report.{ext}', f'{DRIVE}/runs/report_{stamp}.{ext}')
    shutil.copy(f'runs/report.{ext}', f'{DRIVE}/runs/report_latest.{ext}')
print('saved:', f'{DRIVE}/runs/report_{stamp}.md', 'and .html (also report_latest.*)')
display(HTML(open('runs/report.html', encoding='utf-8').read()))

In [ ]:
#@title 8. FINAL - score on the organizers' test set (run ONCE, for the chosen model only; never tune on this)
CONFIRM = False  #@param {type:"boolean"}
import datetime, subprocess
if CONFIRM:
    !python scripts/predict.py --model runs/$RUN/model --input ../data/test/court_cases.jsonl --output runs/$RUN/test_sub.jsonl --batch 32
    out = subprocess.run(['python', '../data/evaluate.py', '../data/test/court_cases.jsonl', f'runs/{RUN}/test_sub.jsonl'], capture_output=True, text=True).stdout
    print(out)
    with open(f'{DRIVE}/runs/test_runs.log', 'a') as f:
        f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M} {RUN} {out.splitlines()[0]}\n")
    !cp runs/$RUN/test_sub.jsonl $DRIVE/runs/$RUN/
else:
    print('set CONFIRM=True to run; this is the final, logged evaluation')